In [0]:
%run ./00_config

## ⚠️ Destructive notebook: run cells individually, never Run all

| Cell | Deletes | Keeps |
|---|---|---|
| Replay | Silver + Gold tables and their checkpoints | Raw files, Bronze |
| Full reset | Raw trades/prices/assets/clients, all tables, all checkpoints | `raw/ticks` |

%md
### Replay: rebuild Silver and Gold from Bronze
After running, run the Job (or `03_silver` then `04_gold`).

In [0]:
SILVER_GOLD_TABLES = ["silver_assets", "silver_clients", "silver_prices", "silver_trades", "silver_ticks",
                      "silver_quarantine", "gold_trade_ledger", "gold_positions",
                      "gold_portfolio_valuation", "gold_client_aum"]
SILVER_GOLD_CHECKPOINTS = ["silver_assets", "silver_clients", "silver_prices", "silver_trades",
                           "silver_ticks", "gold_positions"]

for t in SILVER_GOLD_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{t}")

for c in SILVER_GOLD_CHECKPOINTS:
    dbutils.fs.rm(f"{CHK}/{c}", recurse=True)

print("Replay reset done: Silver and Gold dropped. Bronze and raw files untouched.")

### Full reset: start the whole project from scratch
After running: `01_source_generator` with the backfill loop uncommented (but **not** the test-day `land_day` cells), then the Job.

In [0]:
for f in ["trades", "prices", "assets", "clients"]:
    dbutils.fs.rm(f"{RAW}/{f}", recurse=True)
    dbutils.fs.mkdirs(f"{RAW}/{f}")

dbutils.fs.rm(CHK, recurse=True)
dbutils.fs.mkdirs(CHK)

project_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
                  if r.tableName.startswith(("bronze_", "silver_", "gold_"))]
for t in project_tables:
    spark.sql(f"DROP TABLE {CATALOG}.{SCHEMA}.{t}")

print(f"Full reset done: dropped {len(project_tables)} tables, cleared raw files (except ticks) and all checkpoints.")

### Check: what's left

In [0]:
remaining = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
             if r.tableName.startswith(("bronze_", "silver_", "gold_"))]
print("project tables:", remaining or "none")
print("raw folders:", [f.name for f in dbutils.fs.ls(RAW)])
print("checkpoints:", [f.name for f in dbutils.fs.ls(CHK)] or "none")